# 自定义算子入图 — 在线与离线接入

前面几节介绍了如何构建 AscendIR 图、配置图编译以及认识编译产物。本节继续学习一个常见扩展场景：当 GE 内置算子不能满足计算需求时，如何把自己实现的算子加入图中，并在昇腾设备上执行。

本节使用一个 `AddCustom` 加法算子贯穿讲解。它接收两个相同 shape 的 `DT_FLOAT` Tensor，并输出两者之和。通过这个最小算子，可以把注意力集中在“注册、实现、部署、构图和执行”这条接入链路上。

本节学习大纲如下：

- 自定义算子的作用与基本组成
- 在线与离线两种接入方式
- 自定义算子的完整接入流程
- 在线算子的原型、推导和执行注册
- `EagerExecuteOp` 的运行时职责
- 在线与离线示例的运行方法
- 交付件部署、构图执行与常见问题
- 课后练习

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>本节可运行示例以 CANN 9.0.0 和 Ascend910B4 为环境基线；在线、离线示例均使用 shape 为 [8, 1024] 的 DT_FLOAT 输入，并校验全部输出。</p>
</blockquote>
</div>

## 1. 自定义算子是什么

当模型需要一种 GE 尚未内置的计算，又希望这段计算参与整图编译和 NPU 执行时，可以把它实现为自定义算子。常见场景包括：

- 模型中出现了当前算子库不支持的 op type。
- 已有自研 Ascend C kernel，希望把它接入 GE 图。
- 多个基础计算已融合成一个专用 kernel，希望减少图中的算子数量。

一个可入图的自定义算子通常包含三类信息：

1. **算子原型**：输入、输出、属性以及支持的数据类型。
2. **Host 侧实现**：shape/dtype 推导、tiling 或运行时调度逻辑。
3. **Device 侧实现**：真正运行在 NPU 上的 kernel。

| 维度 | 内置算子 | 自定义算子 |
| --- | --- | --- |
| 能力来源 | 随 CANN 安装 | 由开发者提供 |
| 原型与实现 | 已在算子库中注册 | 需要自行注册并编译 |
| 部署方式 | 使用系统 OPP | 通过自定义 OPP 或 `.so + npubin` 部署 |
| GE 构图 | 直接引用内置 op type | 引用开发者注册的 op type |

自定义算子以独立交付件接入 GE，不需要修改 GE 框架代码。

## 2. 在线与离线两种接入方式

自定义算子和普通 GE 图一样，可以走在线执行，也可以编译成 OM 后离线执行。两种方式的主要区别如下：

| 维度 | 在线接入 | 离线接入 |
| --- | --- | --- |
| 主要接口 | `EagerExecuteOp + Session::RunGraph` | Ascend C V1 算子工程 + ATC + ACL |
| kernel 交付 | 外置 npubin，与 Host `.so` 配套 | 安装到自定义 OPP，由 ATC 选择并写入 OM |
| 图编译时机 | 应用运行时 | 部署前 |
| 执行入口 | GE Session | ACL 模型执行接口 |
| 是否生成 OM | 否 | 是 |

如果应用需要在进程内构图并立即执行，可以选择在线方式；如果部署阶段只希望加载固定的 OM 文件，通常选择标准 Ascend C 算子工程和离线方式。

## 3. 自定义算子的接入流程

无论选择在线还是离线方式，接入过程都可以分为以下五步：

1. 定义算子的输入、输出、属性和数据类型。
2. 实现输出 shape/dtype 推导，确保 GE 能完成图编译和内存规划。
3. 编写并编译 Device kernel。
4. 生成 Host 与 Device 交付件，并配置自定义算子路径。
5. 在 Graph 中创建自定义算子节点，完成编译和执行。

在线自定义算子除了运行时 `Execute` 入口，还可以按需实现编译期推导、在线编译、序列化和 Host 参数更新等能力。各能力接口与调用职责的关系如下：

<p align="center"><img src="./images/capability_interfaces.svg" alt="自定义算子能力接口" width="85%"></p>

<p align="center"><img src="./images/custom_op_e2e.svg" alt="自定义算子入图端到端链路" width="85%"></p>

| 示例 | 构图入口 | kernel 交付方式 | 执行方式 |
| --- | --- | --- | --- |
| 在线示例 | GE Session | `aclrtc` 生成外置 npubin，与 `libcust_opapi.so` 同目录 | `Session::RunGraph` |
| 离线示例 | GE 构图并导出 AIR | 标准 Ascend C V1 OPP 包 | `AIR → ATC → OM → ACL` |

## 4. 算子注册

在线示例需要完成算子原型、输出推导和执行实现三项注册。三项注册使用同一个 op type：`AddCustom`。

### 4.1 REG_OP：定义输入和输出

`REG_OP` 描述算子的接口。构图代码包含算子原型头文件 `add_custom.h` 后，就可以创建 `op::AddCustom` 节点。

```cpp
REG_OP(AddCustom)
    .INPUT(x1, TensorType({DT_FLOAT}))
    .INPUT(x2, TensorType({DT_FLOAT}))
    .OUTPUT(y, TensorType({DT_FLOAT}))
    .OP_END_FACTORY_REG(AddCustom);
```

### 4.2 IMPL_OP：注册 shape 和 dtype 推导

GE 在编译图时需要知道输出 Tensor 的 shape 和 dtype，才能继续完成格式推导和内存规划。`AddCustom` 的两个输入 shape 相同，因此输出 shape 直接取第一个输入，输出类型为 `DT_FLOAT`。

```cpp
graphStatus InferShapeForAdd(gert::InferShapeContext *ctx) {
  *ctx->GetOutputShape(0) = *ctx->GetInputShape(0);
  return GRAPH_SUCCESS;
}

graphStatus InferDataTypeForAdd(gert::InferDataTypeContext *ctx) {
  return ctx->SetOutputDataType(0, DT_FLOAT);
}

IMPL_OP(AddCustom)
    .InferShape(InferShapeForAdd)
    .InferDataType(InferDataTypeForAdd);
```

### 4.3 REG_AUTO_MAPPING_OP：绑定执行实现

`REG_AUTO_MAPPING_OP` 把 `AddCustom` op type 与继承 `EagerExecuteOp` 的 C++ 类关联起来。GE 执行节点时会调用这个类的 `Execute()`。

```cpp
class AddCustom : public EagerExecuteOp { /* ... */ };
REG_AUTO_MAPPING_OP(AddCustom);
```

`REG_OP(AddCustom)`、`IMPL_OP(AddCustom)`、实现类名称以及构图时使用的 op type 必须一致。

### 4.4 离线算子的注册方式

离线示例使用标准 Ascend C V1 算子工程。`OpDef` 声明输入输出并绑定 InferShape、InferDataType 和 Tiling，`OP_ADD` 完成注册。ATC 会从自定义 OPP 包中读取这些信息。

```cpp
class AddCustom : public OpDef {
 public:
  explicit AddCustom(const char *name) : OpDef(name) {
    Input("x").ParamType(REQUIRED).DataType({ge::DT_FLOAT}).Format({ge::FORMAT_ND});
    Input("y").ParamType(REQUIRED).DataType({ge::DT_FLOAT}).Format({ge::FORMAT_ND});
    Output("z").ParamType(REQUIRED).DataType({ge::DT_FLOAT}).Format({ge::FORMAT_ND});
    SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
    AICore().SetTiling(optiling::TilingFunc).AddConfig("ascend910b");
  }
};
OP_ADD(AddCustom);
```

## 5. 在线算子的运行时实现

在线示例通过 `EagerExecuteOp::Execute()` 完成一次自定义算子执行。可以把它理解为 GE 调用自定义 kernel 的入口。

### 5.1 读取输入并申请输出

`Execute()` 先从执行上下文中取得两个输入 Tensor，再按输入的 shape、format、dtype 和字节数申请输出 Tensor。输出内存由 GE 管理，算子只需要通过返回的地址写入结果。

```cpp
graphStatus Execute(gert::EagerOpExecutionContext *ctx) override {
  const auto *x = ctx->GetInputTensor(0);
  const auto *y = ctx->GetInputTensor(1);
  auto *z = ctx->MallocOutputTensor(
      0, x->GetShape(), x->GetFormat(), x->GetDataType(), x->GetSize());
  // 加载并启动 kernel
  return GRAPH_SUCCESS;
}
```

CANN 9.0 的 `MallocOutputTensor` 使用五个参数，最后一个参数是输出字节数。本节源码同时做了参数、shape、dtype 和地址检查，便于在输入不符合 kernel 约束时尽早报错。

> **版本说明**：CANN 9.1 将 `MallocOutputTensor` 改为四参数接口，移除了 `tensor_size`，输出大小由 shape 和 dtype 推导。配套源码通过 `MallocOutputTensorCompat` 的 SFINAE 重载同时适配两版：在 CANN 9.0 中调用五参数接口，在 CANN 9.1 中调用四参数接口。正文保留五参数写法，以保持与本节 CANN 9.0.0 环境基线一致。

### 5.2 加载并启动 kernel

本节的辅助程序先用 `aclrtc` 把 Ascend C 源码编译为 `add_custom_kernel.npubin`。`Execute()` 运行时完成以下操作：

1. 读取 npubin 字节。
2. 调用 `aclrtBinaryLoadFromData` 加载 RTC 生成的 Device ELF。
3. 调用 `aclrtBinaryGetFunction` 获取 `add_custom` 函数。
4. 用 `aclrtLaunchKernelWithHostArgs` 在 GE 提供的 stream 上启动 kernel。
5. 等待 stream 完成后再卸载 binary。

`aclrtLaunchKernelWithHostArgs` 的完整签名包含八个参数：

```cpp
aclError aclrtLaunchKernelWithHostArgs(
    aclrtFuncHandle funcHandle,
    uint32_t numBlocks,
    aclrtStream stream,
    aclrtLaunchKernelCfg *cfg,
    void *hostArgs,
    size_t argsSize,
    aclrtPlaceHolderInfo *placeHolderArray,
    size_t placeHolderNum);
```

本节调用依次传入函数句柄、核数、GE stream、launch 配置、Host 参数地址及大小、占位符数组及数量；不需要 launch 配置和占位符时，分别传入 `nullptr`、`nullptr` 和 `0`。

RTC 产物是内存 Device ELF，因此要使用 `aclrtBinaryLoadFromData`，并设置 `ACL_RT_BINARY_LOAD_OPT_MAGIC`；不能把同一 magic 选项交给 `aclrtBinaryLoadFromFile`。

### 5.3 各部分的职责

| 部分 | 职责 |
| --- | --- |
| `REG_OP` | 定义算子输入输出 |
| `IMPL_OP` | 注册 shape/dtype 推导 |
| `EagerExecuteOp::Execute` | 申请输出并启动 kernel |
| `REG_AUTO_MAPPING_OP` | 让 GE 找到执行实现 |
| `Session::RunGraph` | 触发整图在线编译和执行 |

## 6. 动手实践：运行在线与离线示例

两个示例都使用 `[8, 1024]` 的 `DT_FLOAT` 输入，并校验全部 8192 个输出为 `3.0`。在线示例使用两个值为 `1.5` 的输入，计算 `x + y`；离线示例使用值为 `1.0` 的输入和两个串联节点，计算 `(x + x) + x`。

| 示例 | 源码目录 | 执行链路 |
| --- | --- | --- |
| 在线 | `Sources/03.05/compilable_add_custom/` | Ascend C → aclrtc npubin → EagerExecuteOp → GE Session → NPU |
| 离线 | `Sources/03.05/offline_add_custom/` | Ascend C V1 算子包 → AIR → ATC → OM → ACL |

代码单元会把源码复制到新的 `/tmp` 工作目录后再构建，避免复用旧的 CMake 缓存。运行前请加载 CANN 9.0.0 环境，并确认 0 号设备为 Ascend910B4。

### 6.1 在线示例：EagerExecuteOp + Session::RunGraph

在线示例先用 `aclrtc` 将 Ascend C kernel 编译为 Ascend910B4 的 npubin，再由 `EagerExecuteOp::Execute` 读取输入、申请输出、加载 binary 并启动 kernel。图中包含两个 Data 节点和一个 `AddCustom(data_x, data_y)` 节点，不调用 ATC，也不生成 OM。

成功时会输出 `First element of output: 3`，并出现 `[OK] AddCustom completed GE online compilation and NPU numerical validation`。

In [ ]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

import acl

DEVICE_ID = 0
ACL_SUCCESS = 0

def check_acl(name, ret):
    if ret != ACL_SUCCESS:
        raise RuntimeError("{} 失败，ret={}".format(name, ret))

def detect_soc_version():
    check_acl("acl.init", acl.init())
    selected = False
    try:
        check_acl("acl.rt.set_device", acl.rt.set_device(DEVICE_ID))
        selected = True
        value = acl.get_soc_name()
        if value is None:
            return None
        if isinstance(value, (tuple, list)):
            text_values = [item for item in value if isinstance(item, (str, bytes))]
            value = text_values[-1] if text_values else None
        if value is None:
            return None
        if isinstance(value, bytes):
            value = value.decode("utf-8")
        value = str(value).strip()
        return value if value.startswith("Ascend") else "Ascend" + value
    finally:
        if selected:
            check_acl("acl.rt.reset_device", acl.rt.reset_device(DEVICE_ID))
        check_acl("acl.finalize", acl.finalize())

relative = Path("tutorials/ge_development/03_graph_compilation/Sources/03.05/compilable_add_custom")

def unique_paths(paths):
    result = []
    seen = set()
    for path in paths:
        if path is None:
            continue
        resolved = Path(path).expanduser().resolve()
        text = str(resolved)
        if text not in seen:
            seen.add(text)
            result.append(resolved)
    return result

def repository_roots():
    roots = []
    configured_root = os.environ.get("CANN_LEARNING_HUB_ROOT")
    if configured_root:
        roots.append(Path(configured_root))
    roots.append(Path.home() / "cann-learning-hub")
    try:
        git_root = subprocess.check_output(
            ["git", "-C", str(Path.cwd()), "rev-parse", "--show-toplevel"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if git_root:
            roots.append(Path(git_root))
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass
    roots.extend([Path.cwd(), *Path.cwd().parents])
    return unique_paths(roots)

def locate_sample(relative_path, validator):
    candidates = []
    remembered_root = globals().get("_GE_0305_SOURCE_ROOT")
    if remembered_root:
        candidates.append(Path(remembered_root) / relative_path.name)
    for root in repository_roots():
        candidates.extend([
            root / relative_path,
            root / "tutorials/ge_development/03_graph_compilation/Sources/03.05" / relative_path.name,
            root / "03_graph_compilation/Sources/03.05" / relative_path.name,
            root / "Sources/03.05" / relative_path.name,
        ])
    attempted = unique_paths(candidates)
    for candidate in attempted:
        if validator(candidate):
            return candidate, attempted
    return None, attempted

def is_current_online_sample(candidate):
    required = (
        candidate / "run.sh",
        candidate / "CMakeLists.txt",
        candidate / "kernel_compile/main.cc",
        candidate / "session_run/main.cc",
    )
    if not all(path.is_file() for path in required):
        return False
    cmake_text = (candidate / "CMakeLists.txt").read_text(encoding="utf-8")
    run_text = (candidate / "run.sh").read_text(encoding="utf-8")
    return "COMPILABLE_ADD_BUILD_SESSION_RUN" in cmake_text and "Step 1/3" in run_text

source_sample_dir, attempted_paths = locate_sample(relative, is_current_online_sample)
if source_sample_dir is None:
    attempted = "\n".join("- {}".format(path) for path in attempted_paths)
    raise FileNotFoundError(
        "未找到完整的在线自定义算子样例目录。\n"
        "请确认教程文件包含 03.05 在线样例，或设置 CANN_LEARNING_HUB_ROOT。\n尝试路径：\n{}".format(attempted)
    )
_GE_0305_SOURCE_ROOT = source_sample_dir.parent
workspace_dir = Path(tempfile.mkdtemp(prefix="ge_0305_online_")).resolve()
sample_dir = workspace_dir / "online_add_custom"
shutil.copytree(source_sample_dir, sample_dir, ignore=shutil.ignore_patterns("build", "output", "__pycache__"))
assert os.environ.get("ASCEND_HOME_PATH"), "请先配置 ASCEND_HOME_PATH"
detected = os.environ.get("SOC_VERSION") or detect_soc_version()
soc_version = "Ascend910B4" if not detected else detected
assert soc_version.startswith("Ascend910B"), "本例只支持 Ascend910B，实际为 {}".format(soc_version)
env = os.environ.copy()
env["SOC_VERSION"] = soc_version
print("在线样例源码目录：", source_sample_dir)
print("样例隔离执行目录：", sample_dir)
print("Kernel 目标 SoC：", soc_version)
process = subprocess.Popen(["bash", str(sample_dir / "run.sh")], cwd=sample_dir, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_lines = []
for line in process.stdout:
    log_lines.append(line)
    if line.startswith("[INFO] Step") or "[OK]" in line or "First element" in line:
        print(line, end="")
return_code = process.wait()
if return_code != 0:
    print("[ERROR] 流水线末尾日志：")
    print("".join(log_lines[-80:]))
    raise RuntimeError("在线自定义算子流水线失败，退出码={}".format(return_code))
assert "[OK] AddCustom completed GE online compilation and NPU numerical validation" in "".join(log_lines)
assert "First element of output: 3" in "".join(log_lines)
print("在线样例验证通过")

### 6.2 离线示例：Ascend C V1 算子包 + ATC

离线示例由 `op_host` 注册 `OpDef`、InferShape/InferDataType 与 Tiling，由 `op_kernel` 生成 Ascend910B kernel。构建脚本生成自定义 OPP 包，构图程序导出 AIR，ATC 再把 AIR 编译成 OM，最后由 ACL 加载并执行 OM。

成功时会生成 `single_add.air` 和 `single_add.om`，输出 `First element of output: 3`，并出现 `[OK] AIR -> ATC -> OM -> ACL offline validation`。

In [ ]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

relative = Path("tutorials/ge_development/03_graph_compilation/Sources/03.05/offline_add_custom")

def unique_paths(paths):
    result = []
    seen = set()
    for path in paths:
        if path is None:
            continue
        resolved = Path(path).expanduser().resolve()
        text = str(resolved)
        if text not in seen:
            seen.add(text)
            result.append(resolved)
    return result

def repository_roots():
    roots = []
    configured_root = os.environ.get("CANN_LEARNING_HUB_ROOT")
    if configured_root:
        roots.append(Path(configured_root))
    roots.append(Path.home() / "cann-learning-hub")
    try:
        git_root = subprocess.check_output(
            ["git", "-C", str(Path.cwd()), "rev-parse", "--show-toplevel"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if git_root:
            roots.append(Path(git_root))
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass
    roots.extend([Path.cwd(), *Path.cwd().parents])
    return unique_paths(roots)

def locate_sample(relative_path, validator):
    candidates = []
    remembered_root = globals().get("_GE_0305_SOURCE_ROOT")
    if remembered_root:
        candidates.append(Path(remembered_root) / relative_path.name)
    for root in repository_roots():
        candidates.extend([
            root / relative_path,
            root / "tutorials/ge_development/03_graph_compilation/Sources/03.05" / relative_path.name,
            root / "03_graph_compilation/Sources/03.05" / relative_path.name,
            root / "Sources/03.05" / relative_path.name,
        ])
    attempted = unique_paths(candidates)
    for candidate in attempted:
        if validator(candidate):
            return candidate, attempted
    return None, attempted

def is_current_offline_sample(candidate):
    required = (
        candidate / "run.sh",
        candidate / "CMakeLists.txt",
        candidate / "op_host/add_custom.cpp",
        candidate / "op_kernel/add_custom.cpp",
        candidate / "graph_build/main.cc",
        candidate / "model_exec/main.cc",
    )
    if not all(path.is_file() for path in required):
        return False
    cmake_text = (candidate / "CMakeLists.txt").read_text(encoding="utf-8")
    return "project(offline_add_custom" in cmake_text and "npu_op_package" in cmake_text

source_sample_dir, attempted_paths = locate_sample(relative, is_current_offline_sample)
if source_sample_dir is None:
    attempted = "\n".join("- {}".format(path) for path in attempted_paths)
    raise FileNotFoundError(
        "未找到完整的离线算子包样例目录。请确认教程文件包含 03.05 离线样例，"
        "或设置 CANN_LEARNING_HUB_ROOT。\n尝试路径：\n{}".format(attempted)
    )
_GE_0305_SOURCE_ROOT = source_sample_dir.parent
assert os.environ.get("ASCEND_HOME_PATH"), "请先配置 ASCEND_HOME_PATH"
workspace_dir = Path(tempfile.mkdtemp(prefix="ge_0305_offline_")).resolve()
sample_dir = workspace_dir / "offline_add_custom"
shutil.copytree(source_sample_dir, sample_dir, ignore=shutil.ignore_patterns("build", "output", "__pycache__"))
env = os.environ.copy()
env["SOC_VERSION"] = "Ascend910B4"
print("离线样例源码目录：", source_sample_dir)
print("样例隔离执行目录：", sample_dir)
print("离线模型目标 SoC：", env["SOC_VERSION"])
process = subprocess.Popen(["bash", str(sample_dir / "run.sh")], cwd=sample_dir, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_lines = []
for line in process.stdout:
    log_lines.append(line)
    if line.startswith("[INFO] Step") or "[OK]" in line or "First element" in line:
        print(line, end="")
return_code = process.wait()
if return_code != 0:
    print("[ERROR] 流水线末尾日志：")
    print("".join(log_lines[-80:]))
    raise RuntimeError("离线自定义算子流水线失败，退出码={}".format(return_code))
assert (sample_dir / "output/single_add.air").is_file()
assert (sample_dir / "output/single_add.om").is_file()
assert "[OK] AIR -> ATC -> OM -> ACL offline validation" in "".join(log_lines)
assert "First element of output: 3" in "".join(log_lines)
print("离线样例验证通过")

## 7. 交付件编译与部署

### 7.1 在线交付件

在线方式需要两份配套产物：Host 侧的 `libcust_opapi.so` 和 Device 侧的 `add_custom_kernel.npubin`。CMake 用普通 C++ 编译器生成 `.so`，辅助程序通过 `aclrtc` 为目标 SoC 生成 npubin。

```cmake
add_library(cust_opapi SHARED ge/custom_op.cpp)
target_link_libraries(cust_opapi PRIVATE ascendcl graph register gert dl)

add_executable(kernel_compile kernel_compile/main.cc)
target_link_libraries(kernel_compile PRIVATE ascendcl acl_rtc)
```

```
<custom_opp_root>/
└── op_graph/
    ├── include/
    │   └── add_custom.h
    └── lib/
        └── linux/aarch64/
            ├── libcust_opapi.so
            └── add_custom_kernel.npubin
```

CANN 9.0 会从 `ASCEND_CUSTOM_OPP_PATH` 的路径项中查找 `libcust_opapi.so`。本节把包含 `.so` 和 npubin 的目录直接加入环境变量：

```bash
custom_op_dir="<custom_opp_root>/op_graph/lib/linux/aarch64"
export ASCEND_CUSTOM_OPP_PATH="${custom_op_dir}${ASCEND_CUSTOM_OPP_PATH:+:${ASCEND_CUSTOM_OPP_PATH}}"
```

### 7.2 离线 OPP 算子包

离线示例生成包含 `op_proto`、`op_impl/ai_core/tbe/op_tiling` 和 Ascend910B kernel 的 vendor 包。`ASCEND_CUSTOM_OPP_PATH` 要指向 vendor 根目录，ATC 才能找到完整的算子原型、tiling 和 kernel。

```bash
export ASCEND_CUSTOM_OPP_PATH="<package-root>/vendors/customize${ASCEND_CUSTOM_OPP_PATH:+:${ASCEND_CUSTOM_OPP_PATH}}"
atc --model=single_add.air --framework=1 --output=single_add --soc_version=Ascend910B4
```

## 8. 构图与执行

### 8.1 在 Graph 中创建 AddCustom 节点

在线示例使用两个 Data 节点作为输入，并显式设置输入输出的 `TensorDesc`。随后创建 `op::AddCustom`，连接两个输入，再把它设置为 Graph 输出。

```cpp
TensorDesc desc(Shape({8, 1024}), FORMAT_ND, DT_FLOAT);
auto data_x = op::Data("data_x").set_attr_index(0);
data_x.update_input_desc_x(desc);
data_x.update_output_desc_y(desc);

auto data_y = op::Data("data_y").set_attr_index(1);
data_y.update_input_desc_x(desc);
data_y.update_output_desc_y(desc);

auto add = op::AddCustom("add").set_input_x1(data_x).set_input_x2(data_y);
add.update_input_desc_x1(desc);
add.update_input_desc_x2(desc);
add.update_output_desc_y(desc);

graph.SetInputs({data_x, data_y}).SetOutputs({add});
```

### 8.2 在线执行

设置 `ASCEND_CUSTOM_OPP_PATH` 后，`Session::AddGraph + Session::RunGraph` 会在当前进程内完成图编译和 NPU 执行。两个输入均为 `1.5`，因此 8192 个输出都应为 `3.0`。

```cpp
std::map<AscendString, AscendString> options = {
    {"ge.exec.deviceId", "0"},
    {"ge.graphRunMode", "0"},
};
GEInitialize(options);
Session session(options);
session.AddGraph(0, graph);
session.RunGraph(0, {input_x, input_y}, outputs);
GEFinalize();
```

在线链路不会生成 OM。若 `AddGraph` 或 `RunGraph` 失败，示例会调用 `GEGetErrorMsgV2()` 打印更具体的 GE 错误。

### 8.3 离线执行

离线示例先保存 `single_add.air`，再由 ATC 根据自定义 OPP 包生成 `single_add.om`，最后使用 ACL 加载并执行 OM：

```bash
export SOC_VERSION=Ascend910B4
bash Sources/03.05/offline_add_custom/run.sh
```

## 9. 其他前端如何引用自定义算子

本节使用 GE C++ 算子原型直接构图。若模型来自前端框架，还需要在前端节点与 GE op type 之间增加一层转换。无论前端如何变化，进入 GE 图后的节点仍要能够找到对应的算子原型和实现。

| 前端 | 额外接入工作 |
| --- | --- |
| GE C++ 构图 | 包含算子原型头文件，直接创建 `op::AddCustom` |
| PyTorch + TorchAir | 注册 PyTorch 自定义算子，并编写 FX 节点到 GE op type 的转换器 |
| TensorFlow | 提供 TensorFlow 自定义算子库和支持列表 |
| ONNX | 提供自定义节点解析插件，把 ONNX 节点转换为 GE 节点 |

初次学习时先掌握 GE C++ 原生构图链路，再根据所用框架补充对应的前端注册即可。

## 10. 如何选择在线或离线方式

| 需求 | 推荐方式 | 原因 |
| --- | --- | --- |
| 运行时构图后立即执行 | 在线 | GE Session 在当前进程内完成编译和执行 |
| 部署时只加载固定模型 | 离线 | ATC 提前生成 OM，运行阶段直接用 ACL 加载 |
| 使用外置预编译 kernel 做功能验证 | 在线 | `EagerExecuteOp` 可以直接加载 npubin |
| 标准 Ascend C 算子需要随模型交付 | 离线 | 自定义 OPP 可由 ATC 选择 kernel 并写入 OM |

选择时先确定最终交付物：如果需要 OM，使用标准 Ascend C 算子工程走离线链路；如果不需要 OM，并且希望在 GE Session 中直接执行外置 kernel，可使用在线链路。

## 11. 常见问题定位

| 现象 | 可能原因 | 排查方法 |
| --- | --- | --- |
| 在线 GE 找不到自定义算子 | `ASCEND_CUSTOM_OPP_PATH` 未配置或路径错误 | 确认其中一个路径项直接包含 `libcust_opapi.so` |
| `Execute` 返回 `107000` | RTC Device ELF 使用了错误的加载接口 | 读取 npubin 字节，使用 `aclrtBinaryLoadFromData + MAGIC` |
| kernel 已启动，但 `RunGraph` 仍失败 | 缺少 shape/dtype 推导注册 | 检查 `IMPL_OP(...).InferShape(...).InferDataType(...)`，并查看 `RunGraph GE error:` |
| kernel 偶发失效 | 异步 launch 后立即卸载 binary | `aclrtSynchronizeStream` 成功后再卸载 |
| 在线输入或输出校验失败 | 输入顺序、dtype、shape 或输出字节数不匹配 | 对照 `REG_OP`、`TensorDesc` 和 kernel ABI 逐项检查 |
| ATC 报自定义算子不支持 | 自定义 OPP 路径错误或交付件不完整 | 检查 vendor 根目录中的 `op_proto`、`op_impl`、`op_tiling` 和 kernel |
| npubin 无法执行 | Host/Device 交付件与目标架构或 SoC 不匹配 | 在 CANN 9.0.0、Ascend910B4 环境重新构建 |

### 开发检查清单

- [ ] 算子类型名、原型、推导注册、执行映射保持一致
- [ ] 输入输出描述与 kernel ABI 一致
- [ ] 在线 `.so` 与 npubin 位于正确目录且面向同一目标 SoC
- [ ] 离线 `ASCEND_CUSTOM_OPP_PATH` 指向完整 vendor 根目录
- [ ] 在目标 CANN 和 SoC 环境重新生成 Host/Device 交付件
- [ ] 校验全部输出元素，而不只检查首元素

## 课后练习

本节介绍了自定义算子的基本组成、在线与离线接入流程，以及 CANN 9.0.0 下 `AddCustom` 的注册、部署和执行方法。请完成以下题目进行自测。

1. （判断题）自定义算子必须修改 GE 框架源码后才能加入 AscendIR 图。

2. （判断题）本节在线示例通过 `Session::RunGraph` 执行，不会生成 OM 文件。

3. （单选题）在线示例中，哪个宏用于定义 `AddCustom` 的输入、输出和支持的数据类型？
    A. `IMPL_OP`
    B. `REG_OP`
    C. `OP_ADD`
    D. `REG_AUTO_MAPPING_OP`

4. （单选题）`IMPL_OP(AddCustom)` 在在线链路中的主要作用是什么？
    A. 把 AIR 保存为 OM
    B. 注册输出 shape 和 dtype 推导
    C. 为输入 Tensor 分配 Device 内存
    D. 创建 GE Session

5. （多选题）`EagerExecuteOp::Execute()` 的职责包括哪些？
    A. 获取输入 Tensor
    B. 申请输出 Tensor
    C. 启动 Device kernel
    D. 调用 ATC 生成 OM

6. （单选题）`aclrtc` 生成的内存 Device ELF 应通过哪个接口加载？
    A. `aclrtBinaryLoadFromFile`
    B. `aclrtBinaryLoadFromData`
    C. `aclmdlLoadFromFile`
    D. `aclgrphBuildModel`

7. （多选题）以下关于离线示例的描述，哪些正确？
    A. 使用标准 Ascend C V1 算子工程生成自定义 OPP 包
    B. 先导出 AIR，再由 ATC 生成 OM
    C. 最后由 ACL 加载并执行 OM
    D. 执行时必须由 `EagerExecuteOp` 加载外置 npubin

8. （单选题）离线示例中的 `ASCEND_CUSTOM_OPP_PATH` 应指向哪里？
    A. `single_add.om` 文件
    B. 仅包含 npubin 的目录
    C. 包含 `op_proto`、`op_impl`、`op_tiling` 等内容的 vendor 根目录
    D. Notebook 的 `/tmp` 父目录

**执行以下代码获取答案。**

In [ ]:
!cat answer/03.05_answer.txt